In [1]:
# ============================================================
# 07 / Cell 1
# New-city intake setup: folders + manifest template
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# locate project root
# ------------------------------------------------------------
project_root = Path.cwd()
if not (project_root / "data_processed").exists():
    project_root = project_root.parent

output_dir = project_root / "outputs" / "pilot"
output_dir.mkdir(parents=True, exist_ok=True)

intake_root = project_root / "city_intake"
intake_root.mkdir(parents=True, exist_ok=True)

target_cities = [
    {"city": "Miami", "slug": "miami"},
    {"city": "Las Vegas", "slug": "las_vegas"},
]

# ------------------------------------------------------------
# create standard folders
# ------------------------------------------------------------
for cfg in target_cities:
    slug = cfg["slug"]
    for sub in ["raw", "working", "final"]:
        (intake_root / slug / sub).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# build manifest template
# ------------------------------------------------------------
rows = []
for cfg in target_cities:
    city = cfg["city"]
    slug = cfg["slug"]

    rows.append({
        "city": city,
        "slug": slug,
        "is_active": False,

        # preferred fast path:
        # a merged tract-level gpkg already containing geometry + GEOID + LST + HI + population
        "merged_candidate_gpkg": "",

        # fallback component path:
        "boundary_file": "",
        "tract_geometry_file": "",
        "lst_table_file": "",
        "hi_table_file": "",
        "population_file": "",

        # source columns (fill when known)
        "geoid_col": "GEOID",
        "lst_col_source": "",
        "hi_col_source": "",
        "pop_col_source": "",

        # standard targets
        "target_output_gpkg": str(project_root / "data_processed" / slug / f"{slug}_master_with_lst_hi_fixed.gpkg"),
        "target_boundary_file": str(project_root / "outputs" / "pilot" / "heat_ready" / slug / f"{slug}_boundary_heat_ready.geojson"),

        "notes": "Fill paths first. Preferred: provide merged_candidate_gpkg. Otherwise provide boundary + tract geometry + LST table + HI table (+ population if needed)."
    })

manifest_df = pd.DataFrame(rows)

display(manifest_df)

manifest_out = output_dir / "rq1_new_city_manifest_template.csv"
manifest_df.to_csv(manifest_out, index=False)

readme_text = """# New-city intake workflow

This folder is used to onboard new cities into the RQ1 LST vs Heat Index pipeline.

Recommended fast path:
1. Prepare one merged tract-level GPKG for each new city.
2. The file should contain:
   - geometry
   - GEOID
   - tract-level LST column
   - tract-level HI column
   - population column (preferred)

Fallback path:
- boundary_file
- tract_geometry_file
- lst_table_file
- hi_table_file
- population_file

Current target cities:
- Miami
- Las Vegas
"""

readme_out = intake_root / "README_new_city_intake.md"
readme_out.write_text(readme_text, encoding="utf-8")

print("\nSaved:")
print(manifest_out, "->", manifest_out.exists())
print(readme_out, "->", readme_out.exists())

print("\nCreated folders:")
for cfg in target_cities:
    slug = cfg["slug"]
    print(intake_root / slug / "raw")
    print(intake_root / slug / "working")
    print(intake_root / slug / "final")

,city,slug,is_active,merged_candidate_gpkg,boundary_file,tract_geometry_file,lst_table_file,hi_table_file,population_file,geoid_col,lst_col_source,hi_col_source,pop_col_source,target_output_gpkg,target_boundary_file,notes
0,Miami,miami,False,,,,,,,GEOID,,,,/Users/yufeizhou/Desktop/heat-exposure-compare...,/Users/yufeizhou/Desktop/heat-exposure-compare...,Fill paths first. Preferred: provide merged_ca...
1,Las Vegas,las_vegas,False,,,,,,,GEOID,,,,/Users/yufeizhou/Desktop/heat-exposure-compare...,/Users/yufeizhou/Desktop/heat-exposure-compare...,Fill paths first. Preferred: provide merged_ca...



Saved:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_new_city_manifest_template.csv -> True
/Users/yufeizhou/Desktop/heat-exposure-compare/city_intake/README_new_city_intake.md -> True

Created folders:
/Users/yufeizhou/Desktop/heat-exposure-compare/city_intake/miami/raw
/Users/yufeizhou/Desktop/heat-exposure-compare/city_intake/miami/working
/Users/yufeizhou/Desktop/heat-exposure-compare/city_intake/miami/final
/Users/yufeizhou/Desktop/heat-exposure-compare/city_intake/las_vegas/raw
/Users/yufeizhou/Desktop/heat-exposure-compare/city_intake/las_vegas/working
/Users/yufeizhou/Desktop/heat-exposure-compare/city_intake/las_vegas/final


In [2]:
# ============================================================
# 07 / Cell 2
# Validate new-city manifest and file readiness
# ============================================================

from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if not (project_root / "data_processed").exists():
    project_root = project_root.parent

output_dir = project_root / "outputs" / "pilot"
manifest_path = output_dir / "rq1_new_city_manifest_template.csv"

manifest_df = pd.read_csv(manifest_path, keep_default_na=False)

path_cols = [
    "merged_candidate_gpkg",
    "boundary_file",
    "tract_geometry_file",
    "lst_table_file",
    "hi_table_file",
    "population_file",
]

qc_rows = []

for _, row in manifest_df.iterrows():
    city = row["city"]

    exists_map = {}
    for c in path_cols:
        v = str(row[c]).strip()
        exists_map[c + "_exists"] = (v != "" and Path(v).exists())

    merged_ready = exists_map["merged_candidate_gpkg_exists"]

    component_ready = (
        exists_map["boundary_file_exists"] and
        exists_map["tract_geometry_file_exists"] and
        exists_map["lst_table_file_exists"] and
        exists_map["hi_table_file_exists"]
    )

    if merged_ready:
        readiness = "ready_via_merged_gpkg"
        next_step = "Activate city and standardize columns"
    elif component_ready:
        readiness = "ready_via_components"
        next_step = "Merge components into standard city gpkg"
    else:
        readiness = "missing_inputs"
        missing_list = []
        for c in path_cols:
            v = str(row[c]).strip()
            if c == "population_file":
                # optional in some cases
                continue
            if v == "":
                missing_list.append(c)
            elif not Path(v).exists():
                missing_list.append(c + " (path invalid)")
        next_step = "Missing: " + ", ".join(missing_list) if missing_list else "Need more inputs"

    qc_rows.append({
        "city": city,
        "slug": row["slug"],
        "is_active": row["is_active"],
        "readiness": readiness,
        "next_step": next_step,
        **exists_map
    })

qc_df = pd.DataFrame(qc_rows)

display(qc_df)

qc_out = output_dir / "rq1_new_city_manifest_qc.csv"
qc_df.to_csv(qc_out, index=False)

print("\nSaved:", qc_out)
print(qc_out.exists())

,city,slug,is_active,readiness,next_step,merged_candidate_gpkg_exists,boundary_file_exists,tract_geometry_file_exists,lst_table_file_exists,hi_table_file_exists,population_file_exists
0,Miami,miami,False,missing_inputs,"Missing: merged_candidate_gpkg, boundary_file,...",False,False,False,False,False,False
1,Las Vegas,las_vegas,False,missing_inputs,"Missing: merged_candidate_gpkg, boundary_file,...",False,False,False,False,False,False



Saved: /Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_new_city_manifest_qc.csv
True


In [3]:
# ============================================================
# 07 / Cell 3
# Export target schema reference from ready cities
# ============================================================

from pathlib import Path
import pandas as pd
import geopandas as gpd

project_root = Path.cwd()
if not (project_root / "data_processed").exists():
    project_root = project_root.parent

output_dir = project_root / "outputs" / "pilot"

registry_template = pd.read_csv(output_dir / "rq1_city_registry_template.csv")
ready_df = registry_template.loc[registry_template["status"] == "ready"].copy()

schema_rows = []

for _, row in ready_df.iterrows():
    city = row["city"]
    gpkg_path = Path(row["main_gpkg"])

    gdf = gpd.read_file(gpkg_path)

    for col in gdf.columns:
        dtype_str = str(gdf[col].dtype) if col != "geometry" else "geometry"
        schema_rows.append({
            "city": city,
            "column_name": col,
            "dtype": dtype_str,
            "is_required_core": col in ["GEOID", "lst_c", "hi_c", "total_population", "geometry"]
        })

schema_df = pd.DataFrame(schema_rows)

display(schema_df.head(30))

schema_out = output_dir / "rq1_target_schema_reference_long.csv"
schema_df.to_csv(schema_out, index=False)

# core schema summary
core_cols = ["GEOID", "lst_c", "hi_c", "total_population", "geometry"]
core_summary = []

for col in core_cols:
    sub = schema_df.loc[schema_df["column_name"] == col].copy()
    core_summary.append({
        "column_name": col,
        "present_in_ready_cities_n": len(sub),
        "cities": ", ".join(sorted(sub["city"].unique())) if len(sub) > 0 else "",
        "target_role": {
            "GEOID": "tract join key",
            "lst_c": "tract-level summer LST in Celsius",
            "hi_c": "tract-level summer Heat Index in Celsius",
            "total_population": "tract population for weighted summaries",
            "geometry": "tract polygon geometry"
        }[col]
    })

core_summary_df = pd.DataFrame(core_summary)

display(core_summary_df)

core_summary_out = output_dir / "rq1_target_schema_core.csv"
core_summary_df.to_csv(core_summary_out, index=False)

note_text = """# RQ1 new-city onboarding note

Current batch-ready cities:
- Houston
- Phoenix

Core target schema for any new city:
- GEOID
- lst_c
- hi_c
- total_population
- geometry

Recommended onboarding rule:
Prepare one merged tract-level GPKG per city that already contains the core target schema above. This is the most stable and scalable path for the multicity workflow.
"""

note_out = output_dir / "rq1_new_city_onboarding_note.md"
note_out.write_text(note_text, encoding="utf-8")

print("\nSaved:")
print(schema_out, "->", schema_out.exists())
print(core_summary_out, "->", core_summary_out.exists())
print(note_out, "->", note_out.exists())

,city,column_name,dtype,is_required_core
0,Houston,STATEFP,object,False
1,Houston,COUNTYFP,object,False
2,Houston,TRACTCE,object,False
3,Houston,GEOID,object,True
4,Houston,NAME,object,False
5,Houston,NAMELSAD,object,False
6,Houston,ALAND,int64,False
7,Houston,AWATER,int64,False
8,Houston,area_km2,float64,False
9,Houston,city,object,False


,column_name,present_in_ready_cities_n,cities,target_role
0,GEOID,2,"Houston, Phoenix",tract join key
1,lst_c,2,"Houston, Phoenix",tract-level summer LST in Celsius
2,hi_c,2,"Houston, Phoenix",tract-level summer Heat Index in Celsius
3,total_population,2,"Houston, Phoenix",tract population for weighted summaries
4,geometry,2,"Houston, Phoenix",tract polygon geometry



Saved:
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_target_schema_reference_long.csv -> True
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_target_schema_core.csv -> True
/Users/yufeizhou/Desktop/heat-exposure-compare/outputs/pilot/rq1_new_city_onboarding_note.md -> True
